In [17]:
import sympy as sp
sp.init_printing() # so outputs are printed in LaTeX

In [2]:
PDC1, s, mem, r_m, N, C = sp.symbols('PDC1, s, mem, r_m, N, C')

l_t, d_pdc1 = sp.symbols('l_t, d_pdc1')
R_p, R_s = sp.symbols('R_p, R_s')
g_pdc1, g_s, g_N, g_C = sp.symbols('g_pdc1, g_s, g_N, g_C')
d_N, d_C, d_s = sp.symbols('d_N, d_C, d_s')
k_leak = sp.symbols('k_leak')
k_delay, r_max, r_min = sp.symbols('k_delay, r_max, r_min')
q_pdc, q_N, q_M = sp.symbols('q_pdc, q_N, q_M')
P_max = sp.symbols('P_max')

# Control strain, batch culture, no light (l(t)=0)

In [3]:
cell_logistic = (1 - (N + C)/P_max);

dNdt = ((g_N) * PDC1 - d_N * N) * cell_logistic;
dPDC1dt = ((l_t*g_pdc1) * N) * (1 - PDC1/(N * R_p)) - d_pdc1 * PDC1;
dCdt = ((r_m) * N - d_C * C + g_C * C) * cell_logistic;
dmemdt = (1-l_t) - k_leak*mem;
drmdt = (1/k_delay*mem)*(r_max*(1-l_t) - r_m + r_min);

sysCtrl = sp.Matrix([dPDC1dt, dmemdt, drmdt, dNdt, dCdt])
sysCtrl
# print(r"{}".format(sp.latex(sysCtrl)))

Matrix([
[N*g_pdc1*l_t*(1 - PDC1/(N*R_p)) - PDC1*d_pdc1],
[                        -k_leak*mem - l_t + 1],
[ mem*(-r_m + r_max*(1 - l_t) + r_min)/k_delay],
[      (1 - (C + N)/P_max)*(-N*d_N + PDC1*g_N)],
[ (1 - (C + N)/P_max)*(-C*d_C + C*g_C + N*r_m)]])

In [4]:
sysCtrl_noLight = sysCtrl.subs(l_t, 0) # without light
J_ctrl = sysCtrl_noLight.jacobian([PDC1, mem, r_m, N, C])
J_ctrl

Matrix([
[                -d_pdc1,                              0,                     0,                                                        0,                                                                 0],
[                      0,                        -k_leak,                     0,                                                        0,                                                                 0],
[                      0, (-r_m + r_max + r_min)/k_delay,          -mem/k_delay,                                                        0,                                                                 0],
[g_N*(1 - (C + N)/P_max),                              0,                     0,     -d_N*(1 - (C + N)/P_max) - (-N*d_N + PDC1*g_N)/P_max,                                        -(-N*d_N + PDC1*g_N)/P_max],
[                      0,                              0, N*(1 - (C + N)/P_max), r_m*(1 - (C + N)/P_max) - (-C*d_C + C*g_C + N*r_m)/P_max, (1 - (C + N)/P_max)*(-d_

In [5]:
J_ctrl_trivial = J_ctrl.subs({PDC1:0, N:0, C:0, mem:1/k_leak, r_m:r_max+r_min})
J_ctrl_trivial
# print(r"{}".format(sp.latex(J_ctrl_trivial)))

Matrix([
[-d_pdc1,       0,                   0,             0,          0],
[      0, -k_leak,                   0,             0,          0],
[      0,       0, -1/(k_delay*k_leak),             0,          0],
[    g_N,       0,                   0,          -d_N,          0],
[      0,       0,                   0, r_max + r_min, -d_C + g_C]])

In [6]:
J_ctrl_trivial.eigenvects()
# print(r"{}".format(sp.latex(J_ctrl_trivial.eigenvects())))

[(-d_N,
  1,
  [Matrix([
   [                                0],
   [                                0],
   [                                0],
   [(d_C - d_N - g_C)/(r_max + r_min)],
   [                                1]])]),
 (-d_pdc1,
  1,
  [Matrix([
   [(d_C*d_N - d_C*d_pdc1 - d_N*d_pdc1 - d_N*g_C + d_pdc1**2 + d_pdc1*g_C)/(g_N*r_max + g_N*r_min)],
   [                                                                                             0],
   [                                                                                             0],
   [                                                          (d_C - d_pdc1 - g_C)/(r_max + r_min)],
   [                                                                                             1]])]),
 (-k_leak,
  1,
  [Matrix([
   [0],
   [1],
   [0],
   [0],
   [0]])]),
 (-1/(k_delay*k_leak),
  1,
  [Matrix([
   [0],
   [0],
   [1],
   [0],
   [0]])]),
 (-d_C + g_C,
  1,
  [Matrix([
   [0],
   [0],
   [0],
   [0],
   [1]])])]

In [8]:
J_ctrl_capacity = J_ctrl.subs({PDC1:R_p*N, C:P_max-N, mem:1/k_leak, r_m:r_max+r_min})
J_ctrl_capacity

Matrix([
[-d_pdc1,       0,                   0,                                                                0,                                                                0],
[      0, -k_leak,                   0,                                                                0,                                                                0],
[      0,       0, -1/(k_delay*k_leak),                                                                0,                                                                0],
[      0,       0,                   0,                                       -(N*R_p*g_N - N*d_N)/P_max,                                       -(N*R_p*g_N - N*d_N)/P_max],
[      0,       0,                   0, -(N*(r_max + r_min) - d_C*(-N + P_max) + g_C*(-N + P_max))/P_max, -(N*(r_max + r_min) - d_C*(-N + P_max) + g_C*(-N + P_max))/P_max]])

In [18]:
J_ctrl_capacity.eigenvects()
# print(r"{}".format(sp.latex(J_ctrl_capacity.eigenvects())))

⎡                                                         ⎛                    ↪
⎢⎛      ⎡⎡0 ⎤⎤⎞  ⎛            ⎡⎡1⎤⎤⎞  ⎛           ⎡⎡0⎤⎤⎞  ⎜                    ↪
⎢⎜      ⎢⎢  ⎥⎥⎟  ⎜            ⎢⎢ ⎥⎥⎟  ⎜           ⎢⎢ ⎥⎥⎟  ⎜                    ↪
⎢⎜      ⎢⎢0 ⎥⎥⎟  ⎜            ⎢⎢0⎥⎥⎟  ⎜           ⎢⎢1⎥⎥⎟  ⎜                    ↪
⎢⎜      ⎢⎢  ⎥⎥⎟  ⎜            ⎢⎢ ⎥⎥⎟  ⎜           ⎢⎢ ⎥⎥⎟  ⎜-N⋅Rₚ⋅g_N - N⋅d_C + ↪
⎢⎜0, 1, ⎢⎢0 ⎥⎥⎟, ⎜-d_pdc1, 1, ⎢⎢0⎥⎥⎟, ⎜-kₗₑₐₖ, 1, ⎢⎢0⎥⎥⎟, ⎜─────────────────── ↪
⎢⎜      ⎢⎢  ⎥⎥⎟  ⎜            ⎢⎢ ⎥⎥⎟  ⎜           ⎢⎢ ⎥⎥⎟  ⎜                    ↪
⎢⎜      ⎢⎢-1⎥⎥⎟  ⎜            ⎢⎢0⎥⎥⎟  ⎜           ⎢⎢0⎥⎥⎟  ⎜                    ↪
⎢⎜      ⎢⎢  ⎥⎥⎟  ⎜            ⎢⎢ ⎥⎥⎟  ⎜           ⎢⎢ ⎥⎥⎟  ⎜                    ↪
⎢⎝      ⎣⎣1 ⎦⎦⎠  ⎝            ⎣⎣0⎦⎦⎠  ⎝           ⎣⎣0⎦⎦⎠  ⎜                    ↪
⎣                                                         ⎝                    ↪

↪                                                            ⎡⎡                ↪
↪                          

In [16]:
sp.simplify(-N*R_p*g_N - N*d_C + N*d_N + N*g_C - N*r_max - N*r_min + P_max*d_C - P_max*g_C)


-N*R_p*g_N - N*d_C + N*d_N + N*g_C - N*r_max - N*r_min + P_max*d_C - P_max*g_C

# Chromatin-compacted strain, batch culture, no light (l(t) = 0)

In [19]:
cell_logistic = (1 - (N + C)/P_max);

dNdt = ((g_N) / (g_N*q_N*s + 1) * PDC1 - d_N * N) * cell_logistic;
dsdt = (g_s*N) * (1 - N/(s * R_s)) - d_s*s;
dPDC1dt = ((l_t*g_pdc1) / (l_t*g_pdc1*q_pdc*s + 1) * N) * (1 - PDC1/(N * R_p)) - d_pdc1 * PDC1;
dCdt = ((r_m) / (r_m * q_M * s + 1) * N - d_C * C + g_C * C) * cell_logistic;
dmemdt = (1-l_t) - k_leak*mem;
drmdt = (1/k_delay*mem)*(r_max*(1-l_t) - r_m + r_min);
drmEffdt = (r_m) / (r_m * q_M * s + 1);

sys_sp = sp.Matrix([dPDC1dt, dsdt, dmemdt, drmdt, dNdt, dCdt])
sys_sp
# print(r"{}".format(sp.latex(sys)))

⎡               ⎛    PDC₁⎞                  ⎤
⎢   N⋅g_pdc1⋅lₜ⋅⎜1 - ────⎟                  ⎥
⎢               ⎝    N⋅Rₚ⎠                  ⎥
⎢   ────────────────────── - PDC₁⋅d_pdc1    ⎥
⎢   g_pdc1⋅lₜ⋅q_pdc⋅s + 1                   ⎥
⎢                                           ⎥
⎢              ⎛   N      ⎞                 ⎥
⎢         N⋅gₛ⋅⎜- ──── + 1⎟ - dₛ⋅s          ⎥
⎢              ⎝  Rₛ⋅s    ⎠                 ⎥
⎢                                           ⎥
⎢            -kₗₑₐₖ⋅mem - lₜ + 1            ⎥
⎢                                           ⎥
⎢     mem⋅(-rₘ + rₘₐₓ⋅(1 - lₜ) + rₘᵢₙ)      ⎥
⎢     ────────────────────────────────      ⎥
⎢                 k_delay                   ⎥
⎢                                           ⎥
⎢   ⎛    C + N⎞ ⎛           PDC₁⋅g_N   ⎞    ⎥
⎢   ⎜1 - ─────⎟⋅⎜-N⋅d_N + ─────────────⎟    ⎥
⎢   ⎝    Pₘₐₓ ⎠ ⎝         g_N⋅q_N⋅s + 1⎠    ⎥
⎢                                           ⎥
⎢⎛    C + N⎞ ⎛                     N⋅rₘ    ⎞⎥
⎢⎜1 - ─────⎟⋅⎜-C⋅d_C + C⋅g_C + ───

In [20]:
sysSp_noLight = sys_sp.subs(l_t, 0) # without light

J_sp = sysSp_noLight.jacobian([PDC1, s, mem, r_m, N, C])
J_sp

⎡    -d_pdc1                   0                       0                       ↪
⎢                                                                              ↪
⎢                          2                                                   ↪
⎢                         N ⋅gₛ                                                ↪
⎢       0                 ───── - dₛ                   0                       ↪
⎢                             2                                                ↪
⎢                         Rₛ⋅s                                                 ↪
⎢                                                                              ↪
⎢       0                      0                    -kₗₑₐₖ                     ↪
⎢                                                                              ↪
⎢                                              -rₘ + rₘₐₓ + rₘᵢₙ               ↪
⎢       0                      0               ─────────────────               ↪
⎢                           

In [24]:
J_sp_trivial = J_sp.subs({PDC1:0, N:0, C:0, s:0, mem:1/k_leak, r_m:r_max+r_min})
J_sp_trivial
# print(r"{}".format(sp.latex(J_sp_trivial)))

⎡-d_pdc1   0     0           0             0           0     ⎤
⎢                                                            ⎥
⎢   0     -dₛ    0           0            gₛ           0     ⎥
⎢                                                            ⎥
⎢   0      0   -kₗₑₐₖ        0             0           0     ⎥
⎢                                                            ⎥
⎢                           -1                               ⎥
⎢   0      0     0     ─────────────       0           0     ⎥
⎢                      k_delay⋅kₗₑₐₖ                         ⎥
⎢                                                            ⎥
⎢  g_N     0     0           0           -d_N          0     ⎥
⎢                                                            ⎥
⎣   0      0     0           0        rₘₐₓ + rₘᵢₙ  -d_C + g_C⎦

In [22]:
J_sp_trivial.eigenvects()
# print(r"{}".format(sp.latex(J_sp_trivial)))

⎡                                                        ⎛            ⎡⎡       ↪
⎢                                                        ⎜            ⎢⎢d_C⋅d_ ↪
⎢⎛         ⎡⎡                   0                   ⎤⎤⎞  ⎜            ⎢⎢────── ↪
⎢⎜         ⎢⎢                                       ⎥⎥⎟  ⎜            ⎢⎢       ↪
⎢⎜         ⎢⎢       -d_C⋅gₛ + d_N⋅gₛ + g_C⋅gₛ       ⎥⎥⎟  ⎜            ⎢⎢       ↪
⎢⎜         ⎢⎢───────────────────────────────────────⎥⎥⎟  ⎜            ⎢⎢       ↪
⎢⎜         ⎢⎢d_N⋅rₘₐₓ + d_N⋅rₘᵢₙ - dₛ⋅rₘₐₓ - dₛ⋅rₘᵢₙ⎥⎥⎟  ⎜            ⎢⎢       ↪
⎢⎜         ⎢⎢                                       ⎥⎥⎟  ⎜            ⎢⎢       ↪
⎢⎜         ⎢⎢                   0                   ⎥⎥⎟  ⎜            ⎢⎢       ↪
⎢⎜-d_N, 1, ⎢⎢                                       ⎥⎥⎟, ⎜-d_pdc1, 1, ⎢⎢       ↪
⎢⎜         ⎢⎢                   0                   ⎥⎥⎟  ⎜            ⎢⎢       ↪
⎢⎜         ⎢⎢                                       ⎥⎥⎟  ⎜            ⎢⎢       ↪
⎢⎜         ⎢⎢            d_C

In [25]:
J_sp_otherFp = J_sp.subs({PDC1:0, mem:1/k_leak, r_m:r_max+r_min})
J_sp_otherFp

⎡    -d_pdc1                      0                     0                      ↪
⎢                                                                              ↪
⎢                              2                                               ↪
⎢                             N ⋅gₛ                                            ↪
⎢       0                     ───── - dₛ                0                      ↪
⎢                                 2                                            ↪
⎢                             Rₛ⋅s                                             ↪
⎢                                                                              ↪
⎢       0                         0                   -kₗₑₐₖ                   ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢       0                         0                     0                      ↪
⎢                           

In [ ]:
J_sp_otherFp.eigenvects()